# 02 — Prospective Prediction (Q2)

**Research Question:** Does baseline dopaminergic brain structure predict year 2 PQ-BC severity?

Uses `merge_longitudinal()` to create wide-format data: baseline imaging + year 2 PPS target.

Two datasets: structural-only (12 features) vs combined structural+DTI (14 features).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from copy import deepcopy

from src.core.config import initialize_notebook
env = initialize_notebook(run_name='regression')

## 1. Build longitudinal dataset

In [ ]:
from src.core.preprocessing.ingest import load_and_merge
from src.core.preprocessing.splits import timepoint_split, merge_longitudinal
from src.core.preprocessing.transforms import recode
from src.core.preprocessing.qc import quality_control
from src.core.preprocessing.missing import handle_missing

merged_all = load_and_merge(env)
baseline_df, _ = timepoint_split(env, merged_all)
baseline_df = recode(env, baseline_df)
qc_df, qc_mask = quality_control(env, baseline_df, copy=True)
clean_df = handle_missing(env, qc_df, drop_rows=True)
clean_df = clean_df[clean_df['qc_pass']].copy()
print(f'QC-passed baseline: {len(clean_df)}')

target_col = env.configs.regression['targets'][0]['column']
wide_df = merge_longitudinal(env, clean_df, merged_all, target_col)

# Apply baseline severity cutoff (high-severity cohort, PPS >= 30)
bl_severity_cutoff = env.configs.regression.get('baseline_severity_cutoff', 30)
bl_col = f'{target_col}_baseline'
n_before = len(wide_df)
wide_df = wide_df[wide_df[bl_col] >= bl_severity_cutoff].reset_index(drop=True)
print(f'Baseline severity filter (>= {bl_severity_cutoff}): {n_before} -> {len(wide_df)} subjects')
print(f'Longitudinal dataset: {wide_df.shape}')

## 2. Define two datasets

In [ ]:
from src.core.features import get_roi_columns_from_config

# Structural-only — use raw L/R features to match notebook 01 main result
env_struct = deepcopy(env)
env_struct.configs.data['roi_features']['dopamine_core']['connectivity'] = []
env_struct.configs.regression['feature_transform'] = 'raw'
struct_cols = get_roi_columns_from_config(env_struct.configs.data, ['dopamine_core'])

# Combined (dMRI QC filter)
dmri_col = 'mr_y_qc__incl__dmri_indicator'
combined_wide_df = wide_df[wide_df[dmri_col].astype(str) == '1'].copy().reset_index(drop=True)
env_combined = deepcopy(env)
env_combined.configs.regression['feature_transform'] = 'raw'
combined_cols = get_roi_columns_from_config(env_combined.configs.data, ['dopamine_core'])

print(f'Structural-only: {len(struct_cols)} features, N={len(wide_df)}')
print(f'Combined: {len(combined_cols)} features, N={len(combined_wide_df)}')
print(f'Feature transform: raw (matches notebook 01)')

## 3. Descriptive stats

In [ ]:
bl_col = f'{target_col}_baseline'
y2_col = f'{target_col}_year2'
delta_col = f'{target_col}_delta'

print(f'Baseline PPS: mean={wide_df[bl_col].mean():.1f}, std={wide_df[bl_col].std():.1f}')
print(f'Year 2 PPS:   mean={wide_df[y2_col].mean():.1f}, std={wide_df[y2_col].std():.1f}')

from scipy.stats import pearsonr
r_bl_y2, p_bl_y2 = pearsonr(wide_df[bl_col], wide_df[y2_col])
print(f'\nBaseline-Year2 correlation: r = {r_bl_y2:.3f}, p = {p_bl_y2:.2e}')

In [ ]:
# Baseline vs Year 2 PPS scatter
from scipy import stats as sp_stats

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(wide_df[bl_col], wide_df[y2_col], alpha=0.3, s=20, color='steelblue', edgecolors='navy', lw=0.3)
z = np.polyfit(wide_df[bl_col], wide_df[y2_col], 1); p_line = np.poly1d(z)
xs = np.sort(wide_df[bl_col].values)
ax.plot(xs, p_line(xs), 'r-', lw=2.5, zorder=9)

pstr = 'p < 0.001' if p_bl_y2 < 0.001 else f'p = {p_bl_y2:.4f}'
ax.text(0.05, 0.97, f'r = {r_bl_y2:.3f} ({pstr})\nn = {len(wide_df)}',
        transform=ax.transAxes, fontsize=11, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9), family='monospace')
ax.set_xlabel('Baseline PPS Severity', fontsize=13, fontweight='bold')
ax.set_ylabel('Year 2 PPS Severity', fontsize=13, fontweight='bold')
ax.set_title('PPS Temporal Stability: Baseline vs Year 2', fontsize=14, fontweight='bold')
ax.plot([0, 100], [0, 100], 'k--', alpha=0.3, label='Identity')
ax.legend(fontsize=10)
ax.grid(alpha=0.3, ls='--')
plt.tight_layout()
plt.show()

## 4a. Structural-only: Prospective SVR

In [ ]:
from src.core.regression.pipeline import run_target_with_nested_cv

target_config_q2a = {'name': 'pps_severity_year2', 'column': y2_col}

results_struct_q2 = run_target_with_nested_cv(
    env_struct, wide_df, target_config_q2a, model_name='svr', verbose=True
)
r_struct_q2 = results_struct_q2['svr']['overall']['pearson_r']
n_struct_q2 = results_struct_q2['svr']['n_samples']
print(f'\nStructural-only Q2: r = {r_struct_q2:.4f}, n = {n_struct_q2}')

In [ ]:
# Actual vs Predicted scatter — Q2 structural-only
from scipy.stats import pearsonr
from scipy import stats as sp_stats

if 'y_true' in results_struct_q2['svr']:
    y_true_q2 = results_struct_q2['svr']['y_true']
    y_pred_q2 = results_struct_q2['svr']['y_pred']
else:
    import pickle
    seed = env_struct.configs.run['seed']
    reg_dir = (
        env_struct.repo_root / 'outputs' / env_struct.configs.run['run_name']
        / env_struct.configs.run['run_id'] / f'seed_{seed}' / 'regression'
        / target_config_q2a['name'] / 'svr'
    )
    with open(reg_dir / 'results.pkl', 'rb') as f:
        saved = pickle.load(f)
    folds = saved['svr_folds']
    y_true_q2 = np.concatenate([f['y_test'] for f in folds])
    y_pred_q2 = np.concatenate([f['y_pred'] for f in folds])

rv, pv = pearsonr(y_true_q2, y_pred_q2)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_true_q2, y_pred_q2, alpha=0.4, s=30, color='steelblue', edgecolors='navy', lw=0.5)
z = np.polyfit(y_true_q2, y_pred_q2, 1); p_line = np.poly1d(z)
xs = np.sort(y_true_q2)
ax.plot(xs, p_line(xs), 'r-', lw=2.5, zorder=9)
res = y_pred_q2 - p_line(y_true_q2)
se = np.sqrt(np.sum(res**2) / (len(y_true_q2) - 2))
tcrit = sp_stats.t.ppf(0.975, len(y_true_q2) - 2)
ci = tcrit * se * np.sqrt(1/len(y_true_q2) + (xs - y_true_q2.mean())**2 / np.sum((y_true_q2 - y_true_q2.mean())**2))
ax.fill_between(xs, p_line(xs) - ci, p_line(xs) + ci, color='red', alpha=0.12)

pstr = 'p < 0.001' if pv < 0.001 else f'p = {pv:.4f}'
ax.text(0.05, 0.97, f'r = {rv:.3f} ({pstr})\nn = {len(y_true_q2)}',
        transform=ax.transAxes, fontsize=11, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9), family='monospace')
ax.set_xlabel('Observed Year 2 PPS (residualized)', fontsize=13, fontweight='bold')
ax.set_ylabel('Predicted', fontsize=13, fontweight='bold')
ax.set_title('Prospective SVR — Baseline Brain -> Year 2 PPS (Q2)', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3, ls='--')
plt.tight_layout()
plt.show()

## 4b. Combined: Prospective SVR

In [ ]:
results_combined_q2 = run_target_with_nested_cv(
    env_combined, combined_wide_df, target_config_q2a, model_name='svr', verbose=True
)
r_combined_q2 = results_combined_q2['svr']['overall']['pearson_r']
n_combined_q2 = results_combined_q2['svr']['n_samples']
print(f'\nCombined Q2: r = {r_combined_q2:.4f}, n = {n_combined_q2}')
print(f'Structural-only Q2: r = {r_struct_q2:.4f}, n = {n_struct_q2}')

## 5. Sex-stratified SVR

In [ ]:
from src.core.regression.robustness import sex_stratified_svr

print('--- Structural-only ---')
sex_q2_struct = sex_stratified_svr(env_struct, wide_df, target_config_q2a, model_name='svr')

print('\n--- Combined ---')
sex_q2_combined = sex_stratified_svr(env_combined, combined_wide_df, target_config_q2a, model_name='svr')

## 6. Baseline-filtered prospective prediction

In [ ]:
# NOTE: wide_df is already filtered to baseline PPS >= 30 in cell 3.
# This cell is kept for backward-compat variable names used downstream.
bl_high = wide_df.copy()
print(f'Subjects with baseline PPS >= 30: {len(bl_high)} (= wide_df, already filtered)')
print(f'  Their Y2 scores: mean={bl_high[y2_col].mean():.1f}, std={bl_high[y2_col].std():.1f}')

target_config_q2b = {'name': 'pps_severity_year2_from_bl_high', 'column': y2_col}

results_q2b = run_target_with_nested_cv(
    env_struct, bl_high, target_config_q2b, model_name='svr', verbose=True
)
r_q2b = results_q2b['svr']['overall']['pearson_r']
n_q2b = results_q2b['svr']['n_samples']
print(f'\nQ2b (baseline-high -> Y2): SVR r = {r_q2b:.4f}, n = {n_q2b}')

## 7. Permutation tests

In [ ]:
from src.core.regression.evaluation import permutation_test, compute_permutation_pvalue

# Year 2 permutation test
perm_q2 = permutation_test(
    env_struct, wide_df, target_config_q2a, model_name='svr',
    n_permutations=env_struct.configs.regression['permutation']['n_permutations'], verbose=True,
)
p_emp_q2 = compute_permutation_pvalue(r_struct_q2, perm_q2['null_distribution'])
print(f'\nQ2 Y2 observed r = {r_struct_q2:.4f}, permutation p = {p_emp_q2:.4f}')

## 8. Univariate: AI features -> Year 2 PPS

In [ ]:
from src.core.regression.univariate import (
    extract_bilateral_pairs, prepare_harmonized_data, compute_asymmetry_features,
    univariate_correlations,
)

bilateral_pairs, _ = extract_bilateral_pairs(env_struct.configs.data, ['dopamine_core'])

X_harm, y_harm, df_harm, valid_cols = prepare_harmonized_data(
    wide_df, struct_cols,
    harmonize_config=env_struct.configs.harmonize,
    regression_config=env_struct.configs.regression,
    target_col=y2_col,
    target_name='pps_severity_year2',
    data_config=env_struct.configs.data,
)

valid_pairs = [(n, l, r) for n, l, r in bilateral_pairs
               if l in valid_cols and r in valid_cols]
asym = compute_asymmetry_features(X_harm, valid_cols, valid_pairs)
ai_keys = sorted(k for k in asym if k.endswith('_AI'))

X_ai = np.column_stack([asym[k] for k in ai_keys])
corr_df_q2 = univariate_correlations(X_ai, y_harm, ai_keys, corrections=('bonferroni', 'fdr_bh'))
print('Univariate AI correlations -> Year 2 PPS (FDR + Bonferroni corrected):')
print(corr_df_q2[['feature', 'r', 'p_raw', 'p_bonferroni', 'sig_bonferroni',
                   'p_fdr_bh', 'sig_fdr_bh']].to_string(index=False))

## 9. Compare Q1 vs Q2

In [ ]:
from src.core.regression.evaluation import fisher_z_compare

# Re-run Q1 on the longitudinal subsample for fair comparison
target_config_q1 = env_struct.configs.regression['targets'][0]
results_q1 = run_target_with_nested_cv(
    env_struct, wide_df, target_config_q1, model_name='svr', verbose=False
)
r_q1 = results_q1['svr']['overall']['pearson_r']
n_q1 = results_q1['svr']['n_samples']

print(f'Q1  (cross-sectional):     r = {r_q1:.4f}, n = {n_q1}')
print(f'Q2  (prospective, Y2):     r = {r_struct_q2:.4f}, n = {n_struct_q2}')
print(f'Q2b (BL-filtered -> Y2):   r = {r_q2b:.4f}, n = {n_q2b}')

print('\n--- Fisher z comparisons ---')
for label, r2, n2 in [('Q2-Y2', r_struct_q2, n_struct_q2), ('Q2b-Y2', r_q2b, n_q2b)]:
    if np.isfinite(r2) and n2 > 3:
        z, p_diff = fisher_z_compare(r_q1, n_q1, r2, n2)
        print(f'Q1 vs {label}: Fisher z = {z:.3f}, p = {p_diff:.4f}')

## 10. Year 4 prospective prediction

In [ ]:
# Year 4 prospective: baseline brain -> year 4 PPS
wide_df_y4 = merge_longitudinal(env, clean_df, merged_all, target_col, followup='year4')

# Apply baseline severity cutoff (same as Y2)
bl_col_y4 = f'{target_col}_baseline'
n_before_y4 = len(wide_df_y4)
wide_df_y4 = wide_df_y4[wide_df_y4[bl_col_y4] >= bl_severity_cutoff].reset_index(drop=True)
print(f'Baseline severity filter (>= {bl_severity_cutoff}): {n_before_y4} -> {len(wide_df_y4)} subjects')
print(f'Year 4 longitudinal dataset: {wide_df_y4.shape}')

y4_col = f'{target_col}_year4'
target_config_y4 = {'name': 'pps_severity_year4', 'column': y4_col}

# Structural-only
print('\n--- Structural-only -> Year 4 PPS ---')
results_struct_y4 = run_target_with_nested_cv(
    env_struct, wide_df_y4, target_config_y4, model_name='svr', verbose=True
)
r_struct_y4 = results_struct_y4['svr']['overall']['pearson_r']
n_struct_y4 = results_struct_y4['svr']['n_samples']
print(f'Structural-only Q2 (Y4): r = {r_struct_y4:.4f}, n = {n_struct_y4}')

# Combined
combined_wide_y4 = wide_df_y4[wide_df_y4[dmri_col].astype(str) == '1'].copy().reset_index(drop=True)
print('\n--- Combined -> Year 4 PPS ---')
results_combined_y4 = run_target_with_nested_cv(
    env_combined, combined_wide_y4, target_config_y4, model_name='svr', verbose=True
)
r_combined_y4 = results_combined_y4['svr']['overall']['pearson_r']
n_combined_y4 = results_combined_y4['svr']['n_samples']
print(f'Combined Q2 (Y4): r = {r_combined_y4:.4f}, n = {n_combined_y4}')

# NOTE: bl_high filter is redundant since wide_df_y4 is already baseline-filtered
bl_high_y4 = wide_df_y4.copy()
print(f'\n--- Baseline-filtered Y4 (already applied, n={len(bl_high_y4)}) ---')
if len(bl_high_y4) >= 50:
    target_config_y4b = {'name': 'pps_severity_year4_from_bl_high', 'column': y4_col}
    results_y4b = run_target_with_nested_cv(
        env_struct, bl_high_y4, target_config_y4b, model_name='svr', verbose=True
    )
    r_y4b = results_y4b['svr']['overall']['pearson_r']
    n_y4b = results_y4b['svr']['n_samples']
    print(f'Q2b (baseline-high -> Y4): SVR r = {r_y4b:.4f}, n = {n_y4b}')
else:
    r_y4b, n_y4b = np.nan, 0
    print(f'Too few subjects for baseline-filtered Y4 analysis')

In [ ]:
# Y4 permutation test + Fisher z comparisons (now that Y4 variables are defined)

# Y4 permutation test
if n_struct_y4 > 0:
    perm_q2_y4 = permutation_test(
        env_struct, wide_df_y4, target_config_y4, model_name='svr',
        n_permutations=env_struct.configs.regression['permutation']['n_permutations'], verbose=True,
    )
    p_emp_q2_y4 = compute_permutation_pvalue(r_struct_y4, perm_q2_y4['null_distribution'])
    print(f'\nQ2 Y4 observed r = {r_struct_y4:.4f}, permutation p = {p_emp_q2_y4:.4f}')
else:
    p_emp_q2_y4 = np.nan

# Fisher z: Q1 vs Y4
print('\n--- Fisher z: Q1 vs Year 4 ---')
print(f'Q2  (prospective, Y4):     r = {r_struct_y4:.4f}, n = {n_struct_y4}')
if n_y4b > 0:
    print(f'Q2b (BL-filtered -> Y4):   r = {r_y4b:.4f}, n = {n_y4b}')

comparisons_y4 = [('Q2-Y4', r_struct_y4, n_struct_y4)]
if n_y4b > 0:
    comparisons_y4.append(('Q2b-Y4', r_y4b, n_y4b))

for label, r2, n2 in comparisons_y4:
    if np.isfinite(r2) and n2 > 3:
        z, p_diff = fisher_z_compare(r_q1, n_q1, r2, n2)
        print(f'Q1 vs {label}: Fisher z = {z:.3f}, p = {p_diff:.4f}')

## 11. Cortical thickness Q2 (year 2 and year 4)

In [ ]:
# Cortical thickness prospective: baseline cortical -> year 2 and year 4 PPS
env_cort = deepcopy(env)
env_cort.configs.regression['roi_networks'] = ['cortical_dopamine']

cort_cols = get_roi_columns_from_config(env_cort.configs.data, ['cortical_dopamine'])
cort_present_y2 = [c for c in cort_cols if c in wide_df.columns]
cort_present_y4 = [c for c in cort_cols if c in wide_df_y4.columns]

# Year 2
print(f'Cortical features in Y2 data: {len(cort_present_y2)}')
if len(cort_present_y2) >= 2:
    print('\n--- Cortical -> Year 2 PPS ---')
    results_cort_y2 = run_target_with_nested_cv(
        env_cort, wide_df, target_config_q2a, model_name='svr', verbose=True
    )
    r_cort_y2 = results_cort_y2['svr']['overall']['pearson_r']
    n_cort_y2 = results_cort_y2['svr']['n_samples']
    print(f'Cortical Q2 (Y2): r = {r_cort_y2:.4f}, n = {n_cort_y2}')
else:
    r_cort_y2, n_cort_y2 = np.nan, 0

# Year 4
print(f'\nCortical features in Y4 data: {len(cort_present_y4)}')
if len(cort_present_y4) >= 2:
    print('\n--- Cortical -> Year 4 PPS ---')
    results_cort_y4 = run_target_with_nested_cv(
        env_cort, wide_df_y4, target_config_y4, model_name='svr', verbose=True
    )
    r_cort_y4 = results_cort_y4['svr']['overall']['pearson_r']
    n_cort_y4 = results_cort_y4['svr']['n_samples']
    print(f'Cortical Q2 (Y4): r = {r_cort_y4:.4f}, n = {n_cort_y4}')
else:
    r_cort_y4, n_cort_y4 = np.nan, 0

## 12. Diffusion-only Q2

In [ ]:
# Diffusion-only prospective: SCS MD -> year 2 and year 4 PPS
env_dti = deepcopy(env)
env_dti.configs.data['roi_features']['dopamine_core']['structural'] = []

# Year 2
print('--- Diffusion-only -> Year 2 PPS ---')
results_dti_y2 = run_target_with_nested_cv(
    env_dti, combined_wide_df, target_config_q2a, model_name='svr', verbose=True
)
r_dti_y2 = results_dti_y2['svr']['overall']['pearson_r']
n_dti_y2 = results_dti_y2['svr']['n_samples']
print(f'Diffusion-only Q2 (Y2): r = {r_dti_y2:.4f}, n = {n_dti_y2}')

# Year 4
print('\n--- Diffusion-only -> Year 4 PPS ---')
results_dti_y4 = run_target_with_nested_cv(
    env_dti, combined_wide_y4, target_config_y4, model_name='svr', verbose=True
)
r_dti_y4 = results_dti_y4['svr']['overall']['pearson_r']
n_dti_y4 = results_dti_y4['svr']['n_samples']
print(f'Diffusion-only Q2 (Y4): r = {r_dti_y4:.4f}, n = {n_dti_y4}')

## 13. Per-region Q2

In [ ]:
from src.core.regression.robustness import per_region_svr

# Per-region: structural-only -> year 2 PPS (with FDR-corrected p-values)
print('Per-region SVR -> Year 2 PPS:')
region_df_y2 = per_region_svr(env_struct, wide_df, target_config_q2a, model_name='svr')
print(f'\n{region_df_y2[["region", "r", "p_raw", "p_fdr", "n"]].to_string(index=False)}')

In [ ]:
# Per-region SVR bar chart (Q2 Year 2)
fig, ax = plt.subplots(figsize=(8, max(4, len(region_df_y2) * 0.6)))
ypos = np.arange(len(region_df_y2))
colors = ['#2ca02c' if p < 0.05 else '#ff7f0e' if p < 0.1 else '#d9d9d9'
          for p in region_df_y2['p_fdr']]
ax.barh(ypos, region_df_y2['r'].values, color=colors, edgecolor='black', lw=0.5, alpha=0.85)
ax.set_yticks(ypos)
ax.set_yticklabels(region_df_y2['region'].values, fontsize=11)
ax.set_xlabel('Pearson r (5-fold CV)', fontsize=12, fontweight='bold')
ax.set_title('Per-Region SVR — Baseline Brain -> Y2 PPS (Q2)', fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=1)
for i, row in region_df_y2.iterrows():
    stars = '***' if row['p_fdr'] < 0.001 else '**' if row['p_fdr'] < 0.01 else '*' if row['p_fdr'] < 0.05 else ''
    ax.text(row['r'] + 0.003, i, f'r={row["r"]:.3f} {stars}', va='center', fontsize=10)
ax.grid(axis='x', alpha=0.3, ls='--')
plt.tight_layout()
plt.show()

## 14. Anxiety Q2 (year 2 and year 4)

In [ ]:
# Anxiety prospective: baseline brain -> year 2 and year 4 CBCL anxiety
# NOTE: Same PPS high-severity cohort (baseline PPS >= 30) for discriminant validity
anx_col = 'mh_p_cbcl__dsm__anx_sum'

# Year 2 anxiety
if anx_col in merged_all.columns:
    wide_df_anx_y2 = merge_longitudinal(env, clean_df, merged_all, anx_col, followup='year2')
    # Apply baseline PPS severity cutoff
    pps_bl_col = f'{target_col}_baseline'
    id_col = env.configs.data['columns']['mapping']['id']
    if pps_bl_col not in wide_df_anx_y2.columns:
        bl_tp = env.configs.data['timepoints']['baseline']
        tp_col = env.configs.data['columns']['mapping']['timepoint']
        bl_pps = merged_all[merged_all[tp_col] == bl_tp][[id_col, target_col]].rename(
            columns={target_col: pps_bl_col}
        )
        wide_df_anx_y2 = wide_df_anx_y2.merge(bl_pps, on=id_col, how='left')
    n_before = len(wide_df_anx_y2)
    wide_df_anx_y2 = wide_df_anx_y2[wide_df_anx_y2[pps_bl_col] >= bl_severity_cutoff].reset_index(drop=True)
    print(f'Baseline PPS severity filter (anxiety Y2): {n_before} -> {len(wide_df_anx_y2)} subjects')

    anx_y2_col = f'{anx_col}_year2'
    anx_target_y2 = {'name': 'cbcl_anxiety_year2', 'column': anx_y2_col}

    print('--- Structural-only -> Year 2 Anxiety ---')
    results_anx_y2 = run_target_with_nested_cv(
        env_struct, wide_df_anx_y2, anx_target_y2, model_name='svr', verbose=True
    )
    r_anx_y2 = results_anx_y2['svr']['overall']['pearson_r']
    n_anx_y2 = results_anx_y2['svr']['n_samples']
    print(f'Structural -> Y2 Anxiety: r = {r_anx_y2:.4f}, n = {n_anx_y2}')

    # Year 4 anxiety
    wide_df_anx_y4 = merge_longitudinal(env, clean_df, merged_all, anx_col, followup='year4')
    if pps_bl_col not in wide_df_anx_y4.columns:
        wide_df_anx_y4 = wide_df_anx_y4.merge(bl_pps, on=id_col, how='left')
    n_before_y4 = len(wide_df_anx_y4)
    wide_df_anx_y4 = wide_df_anx_y4[wide_df_anx_y4[pps_bl_col] >= bl_severity_cutoff].reset_index(drop=True)
    print(f'Baseline PPS severity filter (anxiety Y4): {n_before_y4} -> {len(wide_df_anx_y4)} subjects')

    anx_y4_col = f'{anx_col}_year4'
    anx_target_y4 = {'name': 'cbcl_anxiety_year4', 'column': anx_y4_col}

    print('\n--- Structural-only -> Year 4 Anxiety ---')
    results_anx_y4 = run_target_with_nested_cv(
        env_struct, wide_df_anx_y4, anx_target_y4, model_name='svr', verbose=True
    )
    r_anx_y4 = results_anx_y4['svr']['overall']['pearson_r']
    n_anx_y4 = results_anx_y4['svr']['n_samples']
    print(f'Structural -> Y4 Anxiety: r = {r_anx_y4:.4f}, n = {n_anx_y4}')
else:
    print(f'WARNING: {anx_col} not in data')
    r_anx_y2, n_anx_y2, r_anx_y4, n_anx_y4 = np.nan, 0, np.nan, 0

## 15. Summary: Q1 vs Q2-Y2 vs Q2-Y4 across feature sets

In [ ]:
# Summary table: Q2 Year 2 vs Year 4 across feature sets and targets
summary_rows = [
    # PPS Severity
    {'Feature Set': 'Subcortical (12)', 'Target': 'PPS', 'Timepoint': 'Y2', 'r': r_struct_q2, 'n': n_struct_q2},
    {'Feature Set': 'Subcortical (12)', 'Target': 'PPS', 'Timepoint': 'Y4', 'r': r_struct_y4, 'n': n_struct_y4},
    {'Feature Set': 'Subcortical (12)', 'Target': 'PPS (BL>=30)', 'Timepoint': 'Y2', 'r': r_q2b, 'n': n_q2b},
]
if n_y4b > 0:
    summary_rows.append(
        {'Feature Set': 'Subcortical (12)', 'Target': 'PPS (BL>=30)', 'Timepoint': 'Y4', 'r': r_y4b, 'n': n_y4b}
    )
summary_rows.extend([
    {'Feature Set': 'Combined (14)', 'Target': 'PPS', 'Timepoint': 'Y2', 'r': r_combined_q2, 'n': n_combined_q2},
    {'Feature Set': 'Combined (14)', 'Target': 'PPS', 'Timepoint': 'Y4', 'r': r_combined_y4, 'n': n_combined_y4},
    {'Feature Set': 'Cortical (20)', 'Target': 'PPS', 'Timepoint': 'Y2', 'r': r_cort_y2, 'n': n_cort_y2},
    {'Feature Set': 'Cortical (20)', 'Target': 'PPS', 'Timepoint': 'Y4', 'r': r_cort_y4, 'n': n_cort_y4},
    {'Feature Set': 'Diffusion (2)', 'Target': 'PPS', 'Timepoint': 'Y2', 'r': r_dti_y2, 'n': n_dti_y2},
    {'Feature Set': 'Diffusion (2)', 'Target': 'PPS', 'Timepoint': 'Y4', 'r': r_dti_y4, 'n': n_dti_y4},
    # Anxiety
    {'Feature Set': 'Subcortical (12)', 'Target': 'Anxiety', 'Timepoint': 'Y2', 'r': r_anx_y2, 'n': n_anx_y2},
    {'Feature Set': 'Subcortical (12)', 'Target': 'Anxiety', 'Timepoint': 'Y4', 'r': r_anx_y4, 'n': n_anx_y4},
])

summary_df = pd.DataFrame(summary_rows)
summary_df['r'] = summary_df['r'].map(lambda x: f'{x:+.4f}' if pd.notna(x) else 'N/A')
print('Q2 Prospective Summary:')
print(summary_df.to_string(index=False))

In [ ]:
# Summary bar chart: Year 2 vs Year 4 across feature sets (PPS only)
labels = ['Subcortical\n(12)', 'Combined\n(14)', 'Cortical\n(20)', 'Diffusion\n(2)']
y2_rs = [r_struct_q2, r_combined_q2, r_cort_y2, r_dti_y2]
y4_rs = [r_struct_y4, r_combined_y4, r_cort_y4 if 'r_cort_y4' in dir() else np.nan,
         r_dti_y4 if 'r_dti_y4' in dir() else np.nan]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, [r if np.isfinite(r) else 0 for r in y2_rs], width,
               label='Year 2', color='steelblue', edgecolor='black', lw=0.5, alpha=0.85)
bars2 = ax.bar(x + width/2, [r if np.isfinite(r) else 0 for r in y4_rs], width,
               label='Year 4', color='coral', edgecolor='black', lw=0.5, alpha=0.85)

ax.axhline(0, color='black', lw=1)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('Pearson r (5-fold CV)', fontsize=12, fontweight='bold')
ax.set_title('Q2 Prospective: Year 2 vs Year 4 Prediction', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3, ls='--')

for bar, r in zip(bars1, y2_rs):
    if np.isfinite(r):
        ax.text(bar.get_x() + bar.get_width()/2, r + 0.005 * np.sign(r) if r != 0 else 0.005,
                f'{r:.3f}', ha='center', va='bottom' if r >= 0 else 'top', fontsize=9)
for bar, r in zip(bars2, y4_rs):
    if np.isfinite(r):
        ax.text(bar.get_x() + bar.get_width()/2, r + 0.005 * np.sign(r) if r != 0 else 0.005,
                f'{r:.3f}', ha='center', va='bottom' if r >= 0 else 'top', fontsize=9)

plt.tight_layout()
plt.show()

## 16. State marker test: Year 2 cross-sectional (Y2 brain → Y2 PPS)

If the pallidum asymmetry effect is a **state marker** (tracks concurrent symptom state rather than stable trait), we expect:
- **Q1** BL brain → BL PPS: significant (r≈0.16) ✓
- **Q2** BL brain → Y2 PPS: null ✓ (already found)
- **Y2-XS** Y2 brain → Y2 PPS: significant (replicates state marker at Y2)



In [ ]:
# Extract Year 2 rows and apply same QC pipeline as baseline
tp_col = env.configs.data['columns']['mapping']['timepoint']
year2_tp = env.configs.data['timepoints']['year2']

year2_df_raw = merged_all[merged_all[tp_col] == year2_tp].copy()
year2_df_rec = recode(env, year2_df_raw)
year2_qc_df, _ = quality_control(env, year2_df_rec, copy=True)
year2_clean = handle_missing(env, year2_qc_df, drop_rows=True)
year2_clean = year2_clean[year2_clean['qc_pass']].copy().reset_index(drop=True)
print(f'Year 2 rows (all):    {len(year2_df_raw)}')
print(f'Year 2 QC-passed:     {len(year2_clean)}')
print(f'  (attrition from baseline {len(clean_df)}: {len(clean_df) - len(year2_clean)} subjects)')

In [ ]:
# Run SVR: Y2 brain -> Y2 PPS (bin_filter [30,100) now registered in regression.yaml)
target_config_y2xs = {'name': 'pps_severity_y2_crosssectional', 'column': target_col}

print('--- State marker test: Y2 brain -> Y2 PPS (cross-sectional, high-severity) ---')
results_y2xs = run_target_with_nested_cv(
    env_struct, year2_clean, target_config_y2xs, model_name='svr', verbose=True
)
r_y2xs = results_y2xs['svr']['overall']['pearson_r']
n_y2xs = results_y2xs['svr']['n_samples']
print(f'\nY2 cross-sectional: r = {r_y2xs:.4f}, n = {n_y2xs}')

In [ ]:
# Univariate AI correlations for state marker test
# bin_filter applied internally by prepare_harmonized_data via target_name lookup in regression.yaml
from src.core.regression.univariate import (
    extract_bilateral_pairs, prepare_harmonized_data,
    compute_asymmetry_features, univariate_correlations,
)

bilateral_pairs, _ = extract_bilateral_pairs(env_struct.configs.data, ['dopamine_core'])

# Prospective: baseline brain -> Y2 PPS, high-severity at Y2 (bin_filter on pps_severity_year2)
X_harm_q2, y_harm_q2, df_harm_q2, valid_cols_q2 = prepare_harmonized_data(
    wide_df, struct_cols,
    harmonize_config=env_struct.configs.harmonize,
    regression_config=env_struct.configs.regression,
    target_col=y2_col,
    target_name='pps_severity_year2',
    data_config=env_struct.configs.data,
)
valid_pairs_q2 = [(n, l, r) for n, l, r in bilateral_pairs if l in valid_cols_q2 and r in valid_cols_q2]
asym_q2 = compute_asymmetry_features(X_harm_q2, valid_cols_q2, valid_pairs_q2)
ai_keys_q2 = sorted(k for k in asym_q2 if k.endswith('_AI'))
X_ai_q2 = np.column_stack([asym_q2[k] for k in ai_keys_q2])
corr_q2 = univariate_correlations(X_ai_q2, y_harm_q2, ai_keys_q2, corrections=('bonferroni', 'fdr_bh'))

# Y2 cross-sectional: Y2 brain -> Y2 PPS, high-severity at Y2 (bin_filter on pps_severity_y2_crosssectional)
X_harm_y2xs, y_harm_y2xs, df_harm_y2xs, valid_cols_y2xs = prepare_harmonized_data(
    year2_clean, struct_cols,
    harmonize_config=env_struct.configs.harmonize,
    regression_config=env_struct.configs.regression,
    target_col=target_col,
    target_name='pps_severity_y2_crosssectional',
    data_config=env_struct.configs.data,
)
valid_pairs_y2xs = [(n, l, r) for n, l, r in bilateral_pairs if l in valid_cols_y2xs and r in valid_cols_y2xs]
asym_y2xs = compute_asymmetry_features(X_harm_y2xs, valid_cols_y2xs, valid_pairs_y2xs)
ai_keys_y2xs = sorted(k for k in asym_y2xs if k.endswith('_AI'))
X_ai_y2xs = np.column_stack([asym_y2xs[k] for k in ai_keys_y2xs])
corr_y2xs = univariate_correlations(X_ai_y2xs, y_harm_y2xs, ai_keys_y2xs, corrections=('bonferroni', 'fdr_bh'))

print(f'Prospective univariate (BL brain -> Y2 PPS, high-severity): n={len(y_harm_q2)}')
print(corr_q2[['feature', 'r', 'p_raw', 'p_bonferroni', 'sig_bonferroni', 'p_fdr_bh', 'sig_fdr_bh']].to_string(index=False))

print(f'\nY2 cross-sectional univariate (Y2 brain -> Y2 PPS, high-severity): n={len(y_harm_y2xs)}')
print(corr_y2xs[['feature', 'r', 'p_raw', 'p_bonferroni', 'sig_bonferroni', 'p_fdr_bh', 'sig_fdr_bh']].to_string(index=False))

In [ ]:
# Side-by-side: pallidum AI at Q1 (known) vs Q2 prospective vs Y2 cross-sectional
fig, ax = plt.subplots(figsize=(10, 5))

regions = [k.replace('_AI', '') for k in ai_keys_q2]
x = np.arange(len(regions))
w = 0.28

# Q1 pallidum AI values from notebook 01 (known results, raw transform)
q1_ai_r = {'pallidum': -0.154, 'thalamus': -0.110, 'putamen': -0.067,
            'accumbens': -0.062, 'caudate': -0.041, 'VEDC_VTA': -0.009}

r_q1_ai  = np.array([q1_ai_r.get(reg, np.nan) for reg in regions])
r_q2_ai  = corr_q2.set_index('feature').reindex([k for k in ai_keys_q2])['r'].values
r_y2xs_ai = corr_y2xs.set_index('feature').reindex([k for k in ai_keys_y2xs])['r'].values

p_q2_fdr   = corr_q2.set_index('feature').reindex([k for k in ai_keys_q2])['p_fdr_bh'].values
p_y2xs_fdr = corr_y2xs.set_index('feature').reindex([k for k in ai_keys_y2xs])['p_fdr_bh'].values

b1 = ax.bar(x - w, r_q1_ai,  w, label=f'Q1: BL brain→BL PPS (n≈474)', color='#2c7bb6', edgecolor='black', lw=0.5, alpha=0.85)
b2 = ax.bar(x,     r_q2_ai,  w, label=f'Q2: BL brain→Y2 PPS (n={len(y_harm_q2)})',  color='#d7191c', edgecolor='black', lw=0.5, alpha=0.85)
b3 = ax.bar(x + w, r_y2xs_ai, w, label=f'Y2-XS: Y2 brain→Y2 PPS (n={len(y_harm_y2xs)})', color='#1a9641', edgecolor='black', lw=0.5, alpha=0.85)

# Mark FDR-significant bars
for i, (p, r) in enumerate(zip(p_q2_fdr, r_q2_ai)):
    if p < 0.05:
        ax.text(x[i], r - 0.005 if r < 0 else r + 0.003, '*', ha='center', fontsize=13, color='#d7191c')
for i, (p, r) in enumerate(zip(p_y2xs_fdr, r_y2xs_ai)):
    if p < 0.05:
        ax.text(x[i] + w, r - 0.005 if r < 0 else r + 0.003, '*', ha='center', fontsize=13, color='#1a9641')

ax.axhline(0, color='black', lw=1)
ax.set_xticks(x)
ax.set_xticklabels(regions, fontsize=11)
ax.set_ylabel('Pearson r (univariate AI correlation)', fontsize=12, fontweight='bold')
ax.set_title('Pallidum Asymmetry: State Marker Test\nUnvariate AI → PPS Severity (FDR * = p<0.05)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3, ls='--')
plt.tight_layout()
plt.show()

In [ ]:
# State marker comparison: Q1 vs Y2-XS vs Q2
print('━' * 60)
print('  STATE MARKER TEST SUMMARY')
print('━' * 60)
print(f'  Q1   BL brain -> BL PPS  (same-TP):  r = {r_q1:+.4f}, n = {n_q1}')
print(f'  Y2XS Y2 brain -> Y2 PPS  (same-TP):  r = {r_y2xs:+.4f}, n = {n_y2xs}')
print(f'  Q2   BL brain -> Y2 PPS  (cross-TP): r = {r_struct_q2:+.4f}, n = {n_struct_q2}')
print('━' * 60)

from src.core.regression.evaluation import fisher_z_compare

# Q1 vs Y2-XS: is same-timepoint effect consistent across waves?
if np.isfinite(r_y2xs) and n_y2xs > 3:
    z_q1_y2xs, p_q1_y2xs = fisher_z_compare(r_q1, n_q1, r_y2xs, n_y2xs)
    print(f'\nFisher z (Q1 vs Y2-XS): z = {z_q1_y2xs:.3f}, p = {p_q1_y2xs:.4f}')
    if p_q1_y2xs > 0.05:
        print('  -> Q1 ≈ Y2-XS: brain-symptom association replicates at Year 2 (consistent with state marker)')
    else:
        print('  -> Q1 ≠ Y2-XS: association differs across waves')

    # Y2-XS vs Q2: same-TP vs cross-TP at year 2
    z_y2xs_q2, p_y2xs_q2 = fisher_z_compare(r_y2xs, n_y2xs, r_struct_q2, n_struct_q2)
    print(f'Fisher z (Y2-XS vs Q2): z = {z_y2xs_q2:.3f}, p = {p_y2xs_q2:.4f}')
    if p_y2xs_q2 < 0.05:
        print('  -> Y2-XS > Q2: concurrent brain-symptom link stronger than prospective (supports state marker)')

# Per-region Y2 cross-sectional
print('\n--- Per-region SVR: Y2 brain -> Y2 PPS ---')
region_df_y2xs = per_region_svr(env_struct, year2_clean, target_config_y2xs, model_name='svr')
print(region_df_y2xs[['region', 'r', 'p_raw', 'p_fdr', 'n']].to_string(index=False))

In [ ]:
# State marker visualization: 3-bar comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Q1 vs Y2-XS vs Q2 bar chart
ax = axes[0]
labels_sm = ['Q1\nBL→BL PPS\n(same-TP)', 'Y2-XS\nY2→Y2 PPS\n(same-TP)', 'Q2\nBL→Y2 PPS\n(cross-TP)']
rs_sm = [r_q1, r_y2xs, r_struct_q2]
ns_sm = [n_q1, n_y2xs, n_struct_q2]
colors_sm = ['#2c7bb6', '#1a9641', '#d7191c']

bars = ax.bar(labels_sm, rs_sm, color=colors_sm, edgecolor='black', lw=0.8, alpha=0.85, width=0.5)
ax.axhline(0, color='black', lw=1)
for bar, r, n in zip(bars, rs_sm, ns_sm):
    yoff = 0.006 if r >= 0 else -0.006
    va = 'bottom' if r >= 0 else 'top'
    ax.text(bar.get_x() + bar.get_width()/2, r + yoff, f'r={r:+.3f}\nn={n}',
            ha='center', va=va, fontsize=10, fontweight='bold')
ax.set_ylabel('Pearson r (5-fold CV)', fontsize=12, fontweight='bold')
ax.set_title('State Marker Test\nSame-TP vs Cross-TP Brain–Symptom Link', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, ls='--')
ax.set_ylim(min(rs_sm) - 0.06, max(rs_sm) + 0.06)

# Right: per-region comparison BL vs Y2 cross-sectional
ax2 = axes[1]
# Retrieve Q1 per-region from notebook 01 if available, otherwise use known values
q1_region_r = {'pallidum': 0.155, 'thalamus': 0.099, 'putamen': 0.094,
               'accumbens': 0.088, 'caudate': 0.056, 'VEDC_VTA': 0.016}
regions_ordered = region_df_y2xs['region'].tolist()
y2xs_r = region_df_y2xs.set_index('region')['r'].reindex(regions_ordered).values
bl_r = np.array([q1_region_r.get(reg, np.nan) for reg in regions_ordered])

x2 = np.arange(len(regions_ordered))
w = 0.35
b1 = ax2.bar(x2 - w/2, bl_r, w, label='BL brain→BL PPS (Q1)', color='#2c7bb6', edgecolor='black', lw=0.5, alpha=0.85)
b2 = ax2.bar(x2 + w/2, y2xs_r, w, label='Y2 brain→Y2 PPS (Y2-XS)', color='#1a9641', edgecolor='black', lw=0.5, alpha=0.85)
ax2.axhline(0, color='black', lw=1)
ax2.set_xticks(x2)
ax2.set_xticklabels(regions_ordered, rotation=30, ha='right', fontsize=10)
ax2.set_ylabel('Pearson r', fontsize=12, fontweight='bold')
ax2.set_title('Per-Region: BL vs Y2 Cross-Sectional', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3, ls='--')

plt.suptitle('State Marker Analysis: Dopaminergic Brain–PPS Association', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()